# Clustering Algorithms Comparison and Benchmarking

This notebook compares the performance of three clustering algorithms on Grafana logs:
- **K-Means**: Partitioning-based clustering
- **Hierarchical (Agglomerative)**: Hierarchical clustering
- **DBSCAN**: Density-based clustering

## Comparison Metrics:
1. **Quality Metrics**:
   - Silhouette Score (higher is better, range: [-1, 1])
   - Davies-Bouldin Index (lower is better)
   - Calinski-Harabasz Score (higher is better)

2. **Performance Metrics**:
   - Execution time
   - Memory usage
   - Scalability

3. **Practical Metrics**:
   - Number of clusters found
   - Cluster size distribution
   - Noise handling (for DBSCAN)

## Analysis:
- Comprehensive comparison across all metrics
- Visualization of algorithm strengths and weaknesses
- Recommendations based on use case

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully!")

## 1. Load Results from All Algorithms

In [ ]:
# Load metrics from each algorithm
with open('kmeans_metrics.json', 'r') as f:
    kmeans_metrics = json.load(f)

with open('hierarchical_metrics.json', 'r') as f:
    hierarchical_metrics = json.load(f)

with open('dbscan_metrics.json', 'r') as f:
    dbscan_metrics = json.load(f)

print("Metrics loaded successfully!")
print("\nK-Means Metrics:")
print(json.dumps(kmeans_metrics, indent=2))
print("\nHierarchical Metrics:")
print(json.dumps(hierarchical_metrics, indent=2))
print("\nDBSCAN Metrics:")
print(json.dumps(dbscan_metrics, indent=2))

In [ ]:
# Load cluster assignments
kmeans_clusters = pd.read_csv('kmeans_cluster_assignments.csv')
hierarchical_clusters = pd.read_csv('hierarchical_cluster_assignments.csv')
dbscan_clusters = pd.read_csv('dbscan_cluster_assignments.csv')

print(f"K-Means clusters shape: {kmeans_clusters.shape}")
print(f"Hierarchical clusters shape: {hierarchical_clusters.shape}")
print(f"DBSCAN clusters shape: {dbscan_clusters.shape}")

## 2. Compare Quality Metrics

In [ ]:
# Create comparison dataframe
comparison_data = {
    'Algorithm': ['K-Means', 'Hierarchical', 'DBSCAN'],
    'Silhouette Score': [
        kmeans_metrics['silhouette_score'],
        hierarchical_metrics['silhouette_score'],
        dbscan_metrics['silhouette_score'] if dbscan_metrics['silhouette_score'] is not None else 0
    ],
    'Davies-Bouldin Index': [
        kmeans_metrics['davies_bouldin_index'],
        hierarchical_metrics['davies_bouldin_index'],
        dbscan_metrics['davies_bouldin_index'] if dbscan_metrics['davies_bouldin_index'] is not None else 0
    ],
    'Calinski-Harabasz Score': [
        kmeans_metrics['calinski_harabasz_score'],
        hierarchical_metrics['calinski_harabasz_score'],
        dbscan_metrics['calinski_harabasz_score'] if dbscan_metrics['calinski_harabasz_score'] is not None else 0
    ],
    'Number of Clusters': [
        kmeans_metrics['n_clusters'],
        hierarchical_metrics['n_clusters'],
        dbscan_metrics['n_clusters']
    ],
    'Clustering Time (s)': [
        kmeans_metrics['clustering_time'],
        hierarchical_metrics['clustering_time'],
        dbscan_metrics['clustering_time']
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\nClustering Algorithm Comparison:")
print("=" * 100)
print(comparison_df.to_string(index=False))
print("=" * 100)

In [ ]:
# Visualize quality metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Silhouette Score (higher is better)
colors = ['#2ecc71' if x == max(comparison_df['Silhouette Score']) else '#3498db' 
          for x in comparison_df['Silhouette Score']]
axes[0, 0].bar(comparison_df['Algorithm'], comparison_df['Silhouette Score'], color=colors)
axes[0, 0].set_ylabel('Silhouette Score', fontsize=12)
axes[0, 0].set_title('Silhouette Score (Higher is Better)', fontsize=14, fontweight='bold')
axes[0, 0].set_ylim([0, max(comparison_df['Silhouette Score']) * 1.2])
for i, v in enumerate(comparison_df['Silhouette Score']):
    axes[0, 0].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Davies-Bouldin Index (lower is better)
colors = ['#2ecc71' if x == min(comparison_df['Davies-Bouldin Index']) else '#3498db' 
          for x in comparison_df['Davies-Bouldin Index']]
axes[0, 1].bar(comparison_df['Algorithm'], comparison_df['Davies-Bouldin Index'], color=colors)
axes[0, 1].set_ylabel('Davies-Bouldin Index', fontsize=12)
axes[0, 1].set_title('Davies-Bouldin Index (Lower is Better)', fontsize=14, fontweight='bold')
axes[0, 1].set_ylim([0, max(comparison_df['Davies-Bouldin Index']) * 1.2])
for i, v in enumerate(comparison_df['Davies-Bouldin Index']):
    axes[0, 1].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Calinski-Harabasz Score (higher is better)
colors = ['#2ecc71' if x == max(comparison_df['Calinski-Harabasz Score']) else '#3498db' 
          for x in comparison_df['Calinski-Harabasz Score']]
axes[1, 0].bar(comparison_df['Algorithm'], comparison_df['Calinski-Harabasz Score'], color=colors)
axes[1, 0].set_ylabel('Calinski-Harabasz Score', fontsize=12)
axes[1, 0].set_title('Calinski-Harabasz Score (Higher is Better)', fontsize=14, fontweight='bold')
for i, v in enumerate(comparison_df['Calinski-Harabasz Score']):
    axes[1, 0].text(i, v + max(comparison_df['Calinski-Harabasz Score'])*0.02, 
                    f'{v:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Clustering Time (lower is better)
colors = ['#2ecc71' if x == min(comparison_df['Clustering Time (s)']) else '#e74c3c' 
          if x == max(comparison_df['Clustering Time (s)']) else '#3498db'
          for x in comparison_df['Clustering Time (s)']]
axes[1, 1].bar(comparison_df['Algorithm'], comparison_df['Clustering Time (s)'], color=colors)
axes[1, 1].set_ylabel('Time (seconds)', fontsize=12)
axes[1, 1].set_title('Clustering Time (Lower is Better)', fontsize=14, fontweight='bold')
for i, v in enumerate(comparison_df['Clustering Time (s)']):
    axes[1, 1].text(i, v + max(comparison_df['Clustering Time (s)'])*0.02, 
                    f'{v:.2f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('clustering_comparison_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Radar Chart Comparison

In [ ]:
# Create radar chart for normalized comparison
from math import pi

# Normalize metrics to 0-1 scale
def normalize(values, reverse=False):
    min_val, max_val = min(values), max(values)
    if max_val == min_val:
        return [0.5] * len(values)
    normalized = [(v - min_val) / (max_val - min_val) for v in values]
    if reverse:  # For metrics where lower is better
        normalized = [1 - n for n in normalized]
    return normalized

# Prepare data
categories = ['Silhouette\nScore', 'Davies-Bouldin\nIndex', 'Calinski-Harabasz\nScore', 'Speed']

# Normalize each metric
silhouette_norm = normalize(comparison_df['Silhouette Score'])
db_norm = normalize(comparison_df['Davies-Bouldin Index'], reverse=True)  # Lower is better
ch_norm = normalize(comparison_df['Calinski-Harabasz Score'])
time_norm = normalize(comparison_df['Clustering Time (s)'], reverse=True)  # Lower is better

# Create radar chart
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='polar')

# Set up angles
angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
angles += angles[:1]

# Plot data for each algorithm
colors = ['#3498db', '#e74c3c', '#2ecc71']
for idx, algorithm in enumerate(comparison_df['Algorithm']):
    values = [silhouette_norm[idx], db_norm[idx], ch_norm[idx], time_norm[idx]]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=algorithm, color=colors[idx])
    ax.fill(angles, values, alpha=0.15, color=colors[idx])

# Customize chart
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=10)
ax.grid(True, linestyle='--', alpha=0.7)
ax.set_title('Clustering Algorithms Performance Comparison\n(Normalized Metrics)', 
             size=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12)

plt.tight_layout()
plt.savefig('clustering_comparison_radar.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Cluster Distribution Comparison

In [ ]:
# Compare cluster size distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# K-Means
kmeans_dist = kmeans_clusters['cluster'].value_counts().sort_index()
axes[0].bar(kmeans_dist.index, kmeans_dist.values, color='#3498db')
axes[0].set_xlabel('Cluster ID', fontsize=12)
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].set_title('K-Means Cluster Distribution', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Hierarchical
hierarchical_dist = hierarchical_clusters['cluster'].value_counts().sort_index()
axes[1].bar(hierarchical_dist.index, hierarchical_dist.values, color='#e74c3c')
axes[1].set_xlabel('Cluster ID', fontsize=12)
axes[1].set_ylabel('Number of Samples', fontsize=12)
axes[1].set_title('Hierarchical Cluster Distribution', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# DBSCAN
dbscan_dist = dbscan_clusters['cluster'].value_counts().sort_index()
colors = ['red' if idx == -1 else '#2ecc71' for idx in dbscan_dist.index]
axes[2].bar(range(len(dbscan_dist)), dbscan_dist.values, color=colors)
axes[2].set_xlabel('Cluster ID (-1 = Noise)', fontsize=12)
axes[2].set_ylabel('Number of Samples', fontsize=12)
axes[2].set_title('DBSCAN Cluster Distribution', fontsize=13, fontweight='bold')
axes[2].set_xticks(range(len(dbscan_dist)))
axes[2].set_xticklabels(dbscan_dist.index)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('clustering_comparison_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate balance metrics
print("\nCluster Balance Analysis:")
print("=" * 80)
for name, dist in [('K-Means', kmeans_dist), ('Hierarchical', hierarchical_dist), 
                    ('DBSCAN', dbscan_dist[dbscan_dist.index != -1])]:
    if len(dist) > 0:
        std_dev = dist.std()
        mean = dist.mean()
        cv = std_dev / mean if mean > 0 else 0  # Coefficient of variation
        print(f"{name}:")
        print(f"  Mean cluster size: {mean:.2f}")
        print(f"  Std deviation: {std_dev:.2f}")
        print(f"  Coefficient of variation: {cv:.4f} {'(well-balanced)' if cv < 0.5 else '(imbalanced)'}")
        print()
print("=" * 80)

## 5. Cluster Purity and Overlap Analysis

In [ ]:
# Analyze how panels and services are distributed across clusters
algorithms = [
    ('K-Means', kmeans_clusters),
    ('Hierarchical', hierarchical_clusters),
    ('DBSCAN', dbscan_clusters[dbscan_clusters['cluster'] != -1])  # Exclude noise
]

print("\nCluster Purity Analysis:")
print("=" * 80)

for name, clusters in algorithms:
    if len(clusters) == 0:
        continue
    
    print(f"\n{name}:")
    print("-" * 80)
    
    # Panel purity: What percentage of each cluster is dominated by a single panel?
    cluster_panel_purity = []
    for cluster_id in clusters['cluster'].unique():
        cluster_data = clusters[clusters['cluster'] == cluster_id]
        if len(cluster_data) > 0:
            top_panel_count = cluster_data['panel_title'].value_counts().iloc[0]
            purity = top_panel_count / len(cluster_data)
            cluster_panel_purity.append(purity)
    
    avg_panel_purity = np.mean(cluster_panel_purity) if cluster_panel_purity else 0
    
    # Service purity
    cluster_service_purity = []
    for cluster_id in clusters['cluster'].unique():
        cluster_data = clusters[clusters['cluster'] == cluster_id]
        if len(cluster_data) > 0:
            top_service_count = cluster_data['service'].value_counts().iloc[0]
            purity = top_service_count / len(cluster_data)
            cluster_service_purity.append(purity)
    
    avg_service_purity = np.mean(cluster_service_purity) if cluster_service_purity else 0
    
    print(f"  Average Panel Purity: {avg_panel_purity:.4f} (1.0 = perfect)")
    print(f"  Average Service Purity: {avg_service_purity:.4f} (1.0 = perfect)")
    print(f"  Interpretation: {'High purity - clusters are homogeneous' if avg_panel_purity > 0.5 else 'Low purity - clusters are heterogeneous'}")

print("\n" + "=" * 80)

## 6. Summary and Recommendations

In [ ]:
# Generate comprehensive summary
print("=" * 100)
print(" " * 30 + "CLUSTERING ALGORITHMS COMPARISON SUMMARY")
print("=" * 100)

print("\n1. QUALITY METRICS WINNER:")
print("-" * 100)

# Silhouette score
silhouette_winner = comparison_df.loc[comparison_df['Silhouette Score'].idxmax(), 'Algorithm']
print(f"   Best Silhouette Score: {silhouette_winner} ({comparison_df['Silhouette Score'].max():.4f})")

# Davies-Bouldin
db_winner = comparison_df.loc[comparison_df['Davies-Bouldin Index'].idxmin(), 'Algorithm']
print(f"   Best Davies-Bouldin Index: {db_winner} ({comparison_df['Davies-Bouldin Index'].min():.4f})")

# Calinski-Harabasz
ch_winner = comparison_df.loc[comparison_df['Calinski-Harabasz Score'].idxmax(), 'Algorithm']
print(f"   Best Calinski-Harabasz Score: {ch_winner} ({comparison_df['Calinski-Harabasz Score'].max():.2f})")

print("\n2. PERFORMANCE METRICS WINNER:")
print("-" * 100)

# Speed
speed_winner = comparison_df.loc[comparison_df['Clustering Time (s)'].idxmin(), 'Algorithm']
print(f"   Fastest Algorithm: {speed_winner} ({comparison_df['Clustering Time (s)'].min():.2f}s)")
print(f"   Slowest Algorithm: {comparison_df.loc[comparison_df['Clustering Time (s)'].idxmax(), 'Algorithm']} ({comparison_df['Clustering Time (s)'].max():.2f}s)")

print("\n3. ALGORITHM CHARACTERISTICS:")
print("-" * 100)

print("\n   K-Means:")
print("     ✓ Pros: Fast, scalable, well-balanced clusters")
print("     ✗ Cons: Requires predefined k, assumes spherical clusters")
print(f"     • Number of clusters: {kmeans_metrics['n_clusters']}")
print(f"     • Best for: Large datasets, known number of clusters, spherical cluster shapes")

print("\n   Hierarchical:")
print("     ✓ Pros: No need to predefine k, produces dendrogram, captures hierarchy")
print("     ✗ Cons: Computationally expensive, sensitive to noise")
print(f"     • Number of clusters: {hierarchical_metrics['n_clusters']}")
print(f"     • Linkage method: {hierarchical_metrics['linkage_method']}")
print(f"     • Best for: Small to medium datasets, hierarchical relationships, exploratory analysis")

print("\n   DBSCAN:")
print("     ✓ Pros: Finds arbitrary-shaped clusters, identifies outliers, no predefined k")
print("     ✗ Cons: Sensitive to parameters, struggles with varying densities")
print(f"     • Number of clusters: {dbscan_metrics['n_clusters']}")
print(f"     • Noise points: {dbscan_metrics['n_noise']} ({dbscan_metrics['noise_ratio']*100:.1f}%)")
print(f"     • Best for: Datasets with outliers, arbitrary cluster shapes, unknown number of clusters")

print("\n4. RECOMMENDATIONS:")
print("-" * 100)

# Determine overall winner based on quality metrics
quality_scores = []
for idx, row in comparison_df.iterrows():
    # Weighted score: silhouette (40%), davies-bouldin (30%), calinski-harabasz (30%)
    score = (silhouette_norm[idx] * 0.4 + 
             db_norm[idx] * 0.3 + 
             ch_norm[idx] * 0.3)
    quality_scores.append(score)

overall_winner_idx = np.argmax(quality_scores)
overall_winner = comparison_df.iloc[overall_winner_idx]['Algorithm']

print(f"\n   OVERALL WINNER: {overall_winner}")
print(f"   (Based on weighted quality metrics)")

print("\n   Use Case Recommendations:")
print("\n   • For production monitoring with known patterns:")
print("     → K-Means (fast, reliable, consistent cluster sizes)")

print("\n   • For exploratory analysis and understanding data hierarchy:")
print("     → Hierarchical Clustering (dendrogram visualization, flexible k)")

print("\n   • For anomaly detection and outlier identification:")
print("     → DBSCAN (explicit noise detection, arbitrary shapes)")

print("\n   • For large-scale deployments (>100k samples):")
print("     → K-Means with Mini-Batch variant")

print("\n" + "=" * 100)

## 7. Save Comparison Results

In [ ]:
# Save comparison dataframe
comparison_df.to_csv('clustering_comparison_summary.csv', index=False)
print("Comparison summary saved to: clustering_comparison_summary.csv")

# Create detailed comparison report
report = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'algorithms_compared': ['K-Means', 'Hierarchical', 'DBSCAN'],
    'metrics': {
        'kmeans': kmeans_metrics,
        'hierarchical': hierarchical_metrics,
        'dbscan': dbscan_metrics
    },
    'winners': {
        'silhouette_score': silhouette_winner,
        'davies_bouldin_index': db_winner,
        'calinski_harabasz_score': ch_winner,
        'speed': speed_winner,
        'overall': overall_winner
    },
    'recommendations': {
        'production_monitoring': 'K-Means',
        'exploratory_analysis': 'Hierarchical',
        'anomaly_detection': 'DBSCAN',
        'large_scale': 'K-Means (Mini-Batch variant)'
    }
}

with open('clustering_comparison_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print("Detailed comparison report saved to: clustering_comparison_report.json")

print("\nAll comparison results saved successfully!")

## 8. Generate Final Comparison Table

In [ ]:
# Create a comprehensive comparison table
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('tight')
ax.axis('off')

# Prepare data
table_data = [
    ['Metric', 'K-Means', 'Hierarchical', 'DBSCAN', 'Winner'],
    ['Silhouette Score↑', 
     f"{kmeans_metrics['silhouette_score']:.4f}",
     f"{hierarchical_metrics['silhouette_score']:.4f}",
     f"{dbscan_metrics['silhouette_score'] if dbscan_metrics['silhouette_score'] else 'N/A'}",
     silhouette_winner],
    ['Davies-Bouldin↓',
     f"{kmeans_metrics['davies_bouldin_index']:.4f}",
     f"{hierarchical_metrics['davies_bouldin_index']:.4f}",
     f"{dbscan_metrics['davies_bouldin_index'] if dbscan_metrics['davies_bouldin_index'] else 'N/A'}",
     db_winner],
    ['Calinski-Harabasz↑',
     f"{kmeans_metrics['calinski_harabasz_score']:.2f}",
     f"{hierarchical_metrics['calinski_harabasz_score']:.2f}",
     f"{dbscan_metrics['calinski_harabasz_score'] if dbscan_metrics['calinski_harabasz_score'] else 'N/A'}",
     ch_winner],
    ['Clustering Time↓',
     f"{kmeans_metrics['clustering_time']:.2f}s",
     f"{hierarchical_metrics['clustering_time']:.2f}s",
     f"{dbscan_metrics['clustering_time']:.2f}s",
     speed_winner],
    ['Number of Clusters',
     str(kmeans_metrics['n_clusters']),
     str(hierarchical_metrics['n_clusters']),
     str(dbscan_metrics['n_clusters']),
     'N/A'],
    ['Samples Processed',
     f"{kmeans_metrics['n_samples']:,}",
     f"{hierarchical_metrics['n_samples']:,}",
     f"{dbscan_metrics['n_samples']:,}",
     'N/A']
]

# Create table
table = ax.table(cellText=table_data, cellLoc='center', loc='center',
                colWidths=[0.25, 0.18, 0.18, 0.18, 0.21])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Style header row
for i in range(5):
    cell = table[(0, i)]
    cell.set_facecolor('#34495e')
    cell.set_text_props(weight='bold', color='white', fontsize=11)

# Highlight winner column
for i in range(1, len(table_data)):
    cell = table[(i, 4)]
    cell.set_facecolor('#2ecc71')
    cell.set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(table_data)):
    for j in range(4):
        cell = table[(i, j)]
        if i % 2 == 0:
            cell.set_facecolor('#ecf0f1')

plt.title('Clustering Algorithms Comparison Table\n', fontsize=16, fontweight='bold', pad=20)
plt.savefig('clustering_comparison_table.png', dpi=300, bbox_inches='tight')
plt.show()

print("Comparison table saved to: clustering_comparison_table.png")